In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
    "./test/modules/whisper_streaming"
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *

In [ ]:
from whisper_online import FasterWhisperASR, OnlineASRProcessor

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)

In [ ]:
asr = FasterWhisperASR("en", MODEL_SIZE)
asr.use_vad()
online = OnlineASRProcessor(asr)

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)
    online.init()

    full_text = ""
    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")
    start_time = time.perf_counter()
    for segment in segment_audio(audio):
        online.insert_audio_chunk(segment)
        _, _, text = online.process_iter()
        full_text += text
    _, _, text = online.finish()
    full_text += text
    end_time = time.perf_counter()
    print(f"\tProcessed time: {end_time - start_time:.2f} seconds")

    return TRNFormat(
        id = flac.stem,
        text = normalize_text_only_en(full_text).upper()
    )

In [ ]:
%%time
data = search_all_ref_and_hyp(
    src, transcriber,lambda x:normalize_text_only_en(x).upper(), 5
)
# CPU times: user 11min 1s, sys: 20.8 s, total: 11min 21s
# Wall time: 2min 18s

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
parse_sclite_summary(output)